# Fabula — Student Edge Model Fine-Tuning & GGUF Export

This notebook fine-tunes an ultra-fast edge SLM (**`meta-llama/Llama-3.2-1B-Instruct`**) in 4-bit on the Fabula **Compact Blueprint DSL** dataset (`blueprint_train.jsonl`, 1,200 samples) using **Unsloth** on a Kaggle free Tesla T4 GPU.

### Optimizations Applied:
1. **Context Window 1024:** Fits the entire compact prompt + 140-token DSL output with minimal VRAM footprint.
2. **Response-Only Masking (`train_on_responses_only`):** Gradients computed strictly on the assistant DSL completion.
3. **Fast Training:** 1,200 short samples (200 tokens each) train in **~35 minutes** on a single T4 GPU.
4. **4-bit Quantized GGUF Export:** Direct export to `Q4_K_M` GGUF (~750 MB) for sub-5-second local CPU execution.

In [ ]:
# Step 1: Install Unsloth and training dependencies
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
print("Unsloth installed successfully!")

In [ ]:
# Step 2: Load Model & Tokenizer in 4-bit
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only

MAX_SEQ_LENGTH = 1024
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"  # Alternative: "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"

print(f"Loading {MODEL_NAME}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # Auto-select Float16 for Tesla T4
    load_in_4bit=True,
)

# Step 3: Attach Parameter-Efficient LoRA Adapters (Rank 16)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("Model loaded and LoRA adapters attached!")

In [ ]:
# Step 4: Load & Format Blueprint Dataset
import json
import os
from datasets import Dataset

def find_file(filename: str) -> str:
    candidates = [
        os.path.join("/kaggle/working", filename),
        os.path.join("data", "processed", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    if os.path.exists("/kaggle/input"):
        for root, _, files in os.walk("/kaggle/input"):
            if filename in files:
                return os.path.join(root, filename)
    raise FileNotFoundError(f"Could not locate '{filename}'.")

def load_dataset(filepath: str) -> Dataset:
    data = []
    with open(filepath, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                formatted = tokenizer.apply_chat_template(
                    item["messages"],
                    tokenize=False,
                    add_generation_prompt=False,
                )
                data.append({"text": formatted})
    return Dataset.from_list(data)

train_path = find_file("blueprint_train.jsonl")
train_data = load_dataset(train_path)
print(f"Loaded {len(train_data)} training examples from {train_path}.")

In [ ]:
# Step 5: Configure SFTTrainer with Response-Only Loss Masking
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="/kaggle/working/fabula_student_output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    warmup_steps=20,
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=25,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    save_strategy="no",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

# Mask prompt tokens so loss is computed ONLY on the assistant DSL response
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)
print("Trainer configured with response-only loss!")

In [ ]:
# Step 6: Train the Student Model
print("Starting fine-tuning...")
trainer.train()
print("Fine-tuning complete!")

In [ ]:
# Step 7: Export Merged 4-bit Quantized GGUF
GGUF_DIR = "/kaggle/working/fabula_student_gguf"
os.makedirs(GGUF_DIR, exist_ok=True)

print(f"Exporting 4-bit Q4_K_M GGUF to {GGUF_DIR}...")
model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method="q4_k_m",
)
print(f"Done! Download the GGUF file from {GGUF_DIR} and place it in your project's models/ folder.")